# Week 19, Day 3: Counting sort

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w019d3.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


**Predict before you run:** what will this print?



In [ ]:
rolls = [3, 6, 1, 3, 3, 6, 2]
counts = [0] * 7
for r in rolls:
    counts[r] += 1
print(counts)
print([face for face in range(7) for _ in range(counts[face])])

In [ ]:
def counting_sort(items, max_value):
    counts = [0] * (max_value + 1)
    for x in items:
        counts[x] += 1
    result = []
    for value, count in enumerate(counts):
        result.extend([value] * count)
    return result

marks = [72, 45, 90, 45, 61, 72, 72, 38, 100, 0]
print(counting_sort(marks, 100))

*Interactive figure on the website: Counting sort tallies the values into boxes, then reads the boxes out in order..*

In [ ]:
def counting_sort_range(items):
    if not items:
        return []
    low, high = min(items), max(items)
    counts = [0] * (high - low + 1)
    for x in items:
        counts[x - low] += 1                       # shift into 0 .. high - low
    result = []
    for offset, count in enumerate(counts):
        result.extend([offset + low] * count)      # shift back
    return result

print(counting_sort_range([3, -2, 5, 0, -2, 4, -7]))

In [ ]:
import random, time

n = 200_000
marks = [random.randint(0, 100) for _ in range(n)]

start = time.perf_counter()
counts = [0] * 101
for m in marks:
    counts[m] += 1
by_counting = [v for v in range(101) for _ in range(counts[v])]
t1 = time.perf_counter() - start

start = time.perf_counter()
by_sorted = sorted(marks)
t2 = time.perf_counter() - start

print(by_counting == by_sorted, f"counting {t1 * 1000:.0f} ms, sorted {t2 * 1000:.0f} ms")

In [ ]:
def counting_sort_by(records, key, max_key):
    counts = [0] * (max_key + 1)
    for r in records:
        counts[key(r)] += 1
    starts = [0] * (max_key + 1)          # where each key's items begin
    total = 0
    for k in range(max_key + 1):
        starts[k] = total
        total += counts[k]
    result = [None] * len(records)
    for r in records:                     # original order: this is what makes it stable
        k = key(r)
        result[starts[k]] = r
        starts[k] += 1
    return result

students = [("Ada", 3), ("Ben", 1), ("Cy", 3), ("Dee", 2), ("Eve", 1)]
print(counting_sort_by(students, key=lambda s: s[1], max_key=3))

In [ ]:
def radix_sort(numbers):
    if not numbers:
        return []
    result = numbers[:]
    place = 1
    while place <= max(numbers):
        # stable counting sort by the digit in this place (0 to 9)
        counts = [0] * 10
        for x in result:
            counts[x // place % 10] += 1
        starts, total = [0] * 10, 0
        for d in range(10):
            starts[d], total = total, total + counts[d]
        out = [None] * len(result)
        for x in result:
            d = x // place % 10
            out[starts[d]] = x
            starts[d] += 1
        result = out
        place *= 10
    return result

print(radix_sort([170, 45, 75, 90, 802, 24, 2, 66]))

> **Key idea.** Counting sort tallies how many items have each key and writes them out in key order: O(n + k), no comparisons, good when the keys are small whole numbers. The stable version places records using running totals. Radix sort applies stable counting sort to one digit at a time, last digit first.

### Exercise: Count and write out

Write `counting_sort(items, max_value)` for whole numbers from 0 to `max_value`. Don't use `sort`, `sorted` or `count(`.

In [ ]:
# Write your code here


In [ ]:
# Check: run the exercise cell above first, then this one.
check("import random\ndef test_values():\n    \"several\"\n    assert counting_sort([3, 1, 2, 3, 0], 3) == [0, 1, 2, 3, 3]\n    assert counting_sort([], 5) == [] and counting_sort([4], 9) == [4]\n    r = random.Random(1)\n    data = [r.randint(0, 100) for _ in range(50_000)]\n    assert counting_sort(data, 100) == sorted(data)\n    import re\n    assert not re.search(r\"\\bsorted\\(|\\.sort\\(|\\.count\\(\", source)")

### Exercise: Any range

Write `counting_sort_any(items)` that sorts whole numbers of any sign by finding the smallest and largest, and shifting values so the smallest goes in box 0. Return `[]` for an empty list.

In [ ]:
# Write your code here


In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"several\"\n    assert counting_sort_any([3, -2, 5, 0, -2, 4, -7]) == [-7, -2, -2, 0, 3, 4, 5]\n    assert counting_sort_any([]) == [] and counting_sort_any([-5, -5]) == [-5, -5]\n    assert counting_sort_any([1_000_000, 999_998, 999_999]) == [999_998, 999_999, 1_000_000]\n    import re\n    assert not re.search(r\"\\bsorted\\(|\\.sort\\(\", source)")

### Exercise: Records, stably

Write `counting_sort_by(records, key, max_key)` that returns a new list of the records sorted by `key(record)` (whole numbers from 0 to `max_key`), stably, using counts turned into starting positions. Don't use `sort` or `sorted`.

In [ ]:
# Write your code here


In [ ]:
# Check: run the exercise cell above first, then this one.
check("import random\ndef test_values():\n    \"several\"\n    students = [(\"Ada\", 3), (\"Ben\", 1), (\"Cy\", 3), (\"Dee\", 2), (\"Eve\", 1)]\n    assert counting_sort_by(students, lambda s: s[1], 3) == [(\"Ben\", 1), (\"Eve\", 1), (\"Dee\", 2), (\"Ada\", 3), (\"Cy\", 3)]\n    assert counting_sort_by([], lambda s: s, 5) == []\n    r = random.Random(7)\n    recs = [(r.randint(0, 20), i) for i in range(5000)]\n    assert counting_sort_by(recs, lambda p: p[0], 20) == sorted(recs, key=lambda p: p[0])\n    import re\n    assert not re.search(r\"\\bsorted\\(|\\.sort\\(\", source)")

### Exercise: Sorting letters

Write `sort_letters(word)` that returns the lower-case letters of `word` (ignore anything that isn't a letter a to z) in alphabetical order, as a string, using counting sort with 26 boxes (`ord(ch) - ord("a")` gives the box). Then `is_anagram(a, b)` that uses it to check whether two phrases use exactly the same letters.

In [ ]:
# Write your code here


In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"several\"\n    assert sort_letters(\"Banana!\") == \"aaabnn\"\n    assert sort_letters(\"\") == \"\" and sort_letters(\"123\") == \"\"\n    assert is_anagram(\"Listen\", \"Silent\") and is_anagram(\"Dormitory\", \"dirty room!\")\n    assert not is_anagram(\"apple\", \"paper\")\n    import re\n    assert not re.search(r\"\\bsorted\\(|\\.sort\\(\", source)")

### Exercise: Radix sort

Write `radix_sort(numbers)` for whole numbers of 0 or more: stable counting sort on the ones digit, then the tens, and so on, until the place is bigger than the largest number. Also return how many passes it made: return `(sorted_list, passes)`. Don't use `sort` or `sorted`.

In [ ]:
# Write your code here


In [ ]:
# Check: run the exercise cell above first, then this one.
check("import random\ndef test_values():\n    \"several\"\n    assert radix_sort([170, 45, 75, 90, 802, 24, 2, 66]) == ([2, 24, 45, 66, 75, 90, 170, 802], 3)\n    assert radix_sort([]) == ([], 0) and radix_sort([0, 0]) == ([0, 0], 0)\n    assert radix_sort([5, 3]) == ([3, 5], 1)\n    r = random.Random(3)\n    data = [r.randint(0, 10 ** 9) for _ in range(20_000)]\n    out, passes = radix_sort(data)\n    assert out == sorted(data) and passes in (9, 10)\n    import re\n    assert not re.search(r\"\\bsorted\\(|\\.sort\\(\", source)")

### Bug hunt: One box short

This crashes with `IndexError` when a mark of 100 turns up. Fix it.

In [ ]:
def counting_sort(marks):
    counts = [0] * 100
    for m in marks:
        counts[m] += 1
    return [v for v in range(100) for _ in range(counts[v])]

print(counting_sort([72, 100, 45, 0, 100]))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"sorted\"\n    expect_output(\"[0, 45, 72, 100, 100]\")")

### Bug hunt: Radix sort, wrong way round

This radix sort processes digits from the **most** significant to the least, and the result isn't sorted. Change the order of the passes so it prints `[2, 24, 45, 66, 75, 90, 170, 802]`.

In [ ]:
def by_digit(numbers, place):
    buckets = [[] for _ in range(10)]
    for x in numbers:
        buckets[x // place % 10].append(x)
    return [x for b in buckets for x in b]

numbers = [170, 45, 75, 90, 802, 24, 2, 66]
for place in [100, 10, 1]:
    numbers = by_digit(numbers, place)
print(numbers)

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"sorted\"\n    expect_output(\"[2, 24, 45, 66, 75, 90, 170, 802]\")")

### Puzzle: Starting positions

Arrange the lines so `starts` holds where each key's items begin, given `counts`. One line is a trap.

(Put these lines in order on the website.)

### Exercise: Predict them

Work these out without running them.

In [ ]:
a = ___   # the counts list for counting_sort([2, 0, 2, 1], 2)
b = ___   # the starting positions for those counts
c = ___   # the list after ONE radix pass (ones digit) on [21, 13, 32, 11]
d = ___   # how many boxes counting sort needs for whole numbers from -5 to 5

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert a == [1, 1, 2] and b == [0, 1, 2] and c == [21, 11, 32, 13] and d == 11")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: Challenge: sorting words by radix

Radix sort works on strings too. Write `radix_sort_words(words)` that sorts lower-case words (letters a to z only) alphabetically, using stable counting sort on one **character position** at a time, from the last position to the first.

Words can have different lengths: treat a missing character as coming **before** "a" (so "car" comes before "cart"). Use 27 boxes: box 0 for "no character here", boxes 1 to 26 for a to z. Don't use `sort` or `sorted`.

In [ ]:
# Write your code here


In [ ]:
# Check: run the exercise cell above first, then this one.
check("import random, string\ndef test_small():\n    \"small\"\n    assert radix_sort_words([\"cart\", \"car\", \"bat\", \"ant\", \"cat\", \"a\", \"\"]) == [\"\", \"a\", \"ant\", \"bat\", \"car\", \"cart\", \"cat\"]\n    assert radix_sort_words([]) == []\n    assert radix_sort_words([\"b\", \"a\", \"b\"]) == [\"a\", \"b\", \"b\"]\ndef test_random():\n    \"random\"\n    r = random.Random(4)\n    words = [\"\".join(r.choice(\"abcde\") for _ in range(r.randint(0, 6))) for _ in range(3000)]\n    assert radix_sort_words(words) == sorted(words)\n    import re\n    assert not re.search(r\"\\bsorted\\(|\\.sort\\(\", source)")

## Warm-up

The first line tallies how many times each face came up. The second writes each face out as many times as it was counted, in order of the face. The rolls are sorted, and no two rolls were ever compared.

## Counting instead of comparing

**Counting sort**, for whole numbers from 0 up to some maximum `k`:

1. Make a list of `k + 1` counts, all 0: one box for each possible value.
2. Go through the items, adding 1 to the box for each item's value.
3. Go through the boxes in order, writing each value out as many times as its box says.

**Negative numbers, or a range that doesn't start at 0:** shift every value by the smallest one, so the smallest maps to box 0:

## Cost and limits

Counting the items is O(n). Reading out the boxes is O(k) for the boxes plus O(n) for the items written. Total: **O(n + k)** time and O(k) extra space for the boxes.

- When k is small compared to n (marks out of 100 for 100,000 students), that's **O(n)**: faster than any comparison sort can be.
- When k is huge (sorting ten numbers that could be anywhere up to a billion), it's terrible: a billion boxes to make and read, for ten items.

In Python, `sorted` is written in C, so it often still wins on time even when counting sort does less work: one line of Python costs much more than one step in C. In a compiled language, and on enormous inputs with a small range, counting sort's O(n) wins outright. The point of today is the **idea**, which turns up everywhere: in histograms, in radix sort, and in how databases group and order data.

Counting sort also needs the values to **be** small whole numbers, or to be turned into them. You can't count-sort names or decimal measurements directly.

## Sorting records stably

The version above writes out the **values**, which is fine for plain numbers but useless for records: if you're sorting students by mark, you need the students back, not just the marks. And you'll want it **stable**, so students with the same mark keep their original order.

The stable version uses the counts to work out **where each value's items start** in the output:

1. Count the items with each key, as before.
2. Turn the counts into **starting positions**: the items with key 0 start at position 0; key 1 items start right after all the key 0 items; and so on. Each start is the running total of all the counts before it.
3. Go through the items **in their original order**, putting each one at the next free position for its key, and moving that position along by one.

Ben comes before Eve, and Ada before Cy, as they were at the start. Because items are placed in their original order, the first item with each key gets that key's first slot.

## Radix sort

What if the numbers are big, like 9-digit ID numbers? Counting sort would need a billion boxes. **Radix sort** sorts them one **digit** at a time, using stable counting sort with just 10 boxes for each digit, starting from the **last** digit:

Why does sorting by the **last** digit first work? After sorting by the ones digit, then the tens, the numbers are in order by their last two digits, because the tens sort was stable: among numbers with the same tens digit, it kept the order the ones sort had made. Each pass adds one more digit of correct order, and the last pass, on the most significant digit, decides the final order, with ties already settled by the passes before it. This only works because each pass is **stable**.

Radix sort makes d passes, one per digit, each O(n + 10): **O(d × n)** in total. For numbers with a fixed number of digits, that's O(n). It's the same trick you'd use by hand to sort a pile of exam papers by candidate number: sort into ten piles by the last digit, stack them up, then by the next digit, and so on. Punched-card sorting machines, descendants of the tabulators Herman Hollerith built for the 1890 United States census, sorted cards exactly this way for decades.

## Choosing a sort

| Situation | Good choice |
|---|---|
| Anything, in real Python code | `sorted` or `list.sort` |
| Whole numbers in a small range, many of them | counting sort |
| Fixed-length whole numbers or strings, a huge number of them | radix sort |
| Need a guaranteed O(n log n), or stability, or data that doesn't fit in memory | merge sort |
| Need in place and fast on average | quicksort with a random pivot |
| Small or nearly sorted lists | insertion sort |

## Practice

## Quiz

## Daily challenge

## Before you go

Sort a handful of coins by value using counting: make a pile for each coin value, deal the coins into the piles, then pick up the piles in order. Notice that you never compared two coins. Tomorrow: Timsort, the algorithm behind Python's own `sorted`, which combines merge sort and insertion sort and takes advantage of order that's already in your data.